In [0]:
# ──────────────────────────────────────────────────────────────
# CELL 1 │ Configuration & Secrets  (updated SASL_JAAS)
# ──────────────────────────────────────────────────────────────
EH_CONN_STR = dbutils.secrets.get(scope="fraud-platform", key="event-hubs-connection-string")

EH_NAMESPACE    = "fraud-platform-eh.servicebus.windows.net:9093"
TOPIC           = "raw.transactions"
BRONZE_PATH     = "/Volumes/workspace/fraud_platform/data/bronze/transactions"
CHECKPOINT_PATH = "/Volumes/workspace/fraud_platform/data/checkpoints/bronze"

SASL_JAAS = (
    'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
    'username="$ConnectionString" '
    f'password="{EH_CONN_STR}";'
)

print("✅ Config loaded")
print(f"   Broker : {EH_NAMESPACE}")
print(f"   Topic  : {TOPIC}")
print(f"   Bronze : {BRONZE_PATH}")

✅ Config loaded
   Broker : fraud-platform-eh.servicebus.windows.net:9093
   Topic  : raw.transactions
   Bronze : /Volumes/workspace/fraud_platform/data/bronze/transactions


In [0]:
# ──────────────────────────────────────────────────────────────
# CELL 2 │ Create the Kafka read-stream
# ──────────────────────────────────────────────────────────────
# spark-sql-kafka-0-10 is bundled with every Databricks Runtime — no %pip needed
raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers",  EH_NAMESPACE)
    .option("kafka.security.protocol",  "SASL_SSL")
    .option("kafka.sasl.mechanism",     "PLAIN")
    .option("kafka.sasl.jaas.config",   SASL_JAAS)
    .option("subscribe",                TOPIC)
    .option("startingOffsets",          "latest")   # only new messages
    .option("failOnDataLoss",           "false")    # safe for demos; topic retention may drop old msgs
    .load()
)

print("✅ Read-stream created (lazy — nothing flows yet)")
raw_stream.printSchema()

✅ Read-stream created (lazy — nothing flows yet)
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [0]:
# ──────────────────────────────────────────────────────────────
# CELL 3 │ Parse JSON → typed columns
# ──────────────────────────────────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, LongType
)

# Must match exactly what transaction_producer.py produces
TXN_SCHEMA = StructType([
    StructField("transaction_id",         StringType(),  True),
    StructField("card_id",                StringType(),  True),
    StructField("customer_id",            StringType(),  True),
    StructField("merchant_id",            StringType(),  True),
    StructField("merchant_name",          StringType(),  True),
    StructField("merchant_category_code", StringType(),  True),
    StructField("amount_local",           DoubleType(),  True),
    StructField("currency_code",          StringType(),  True),
    StructField("amount_eur",             DoubleType(),  True),
    StructField("country_code",           StringType(),  True),
    StructField("terminal_type",          StringType(),  True),
    StructField("event_timestamp",        LongType(),    True),  # epoch ms from producer
    StructField("ip_address",             StringType(),  True),
    StructField("latitude",               DoubleType(),  True),
    StructField("longitude",              DoubleType(),  True),
    StructField("_fraud_simulation_type", StringType(),  True),  # only on suspicious txns
])

bronze_stream = (
    raw_stream
    # Keep Kafka metadata alongside the payload — useful for debugging / replay
    .select(
        F.col("partition").alias("kafka_partition"),
        F.col("offset").alias("kafka_offset"),
        F.col("timestamp").alias("kafka_timestamp"),
        F.from_json(
            F.col("value").cast("string"), TXN_SCHEMA
        ).alias("txn"),
    )
    .select(
        "kafka_partition",
        "kafka_offset",
        "kafka_timestamp",
        "txn.*",                                                    # flatten all transaction fields
        F.to_timestamp(F.col("txn.event_timestamp") / 1000)        # epoch ms → proper timestamp
          .alias("event_ts"),
        F.current_timestamp().alias("ingested_at"),                 # Bronze arrival time
        F.lit("raw.transactions").alias("source_topic"),
    )
)

print("✅ Bronze stream schema:")
bronze_stream.printSchema()

✅ Bronze stream schema:
root
 |-- kafka_partition: integer (nullable = true)
 |-- kafka_offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- transaction_id: string (nullable = true)
 |-- card_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- merchant_id: string (nullable = true)
 |-- merchant_name: string (nullable = true)
 |-- merchant_category_code: string (nullable = true)
 |-- amount_local: double (nullable = true)
 |-- currency_code: string (nullable = true)
 |-- amount_eur: double (nullable = true)
 |-- country_code: string (nullable = true)
 |-- terminal_type: string (nullable = true)
 |-- event_timestamp: long (nullable = true)
 |-- ip_address: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- _fraud_simulation_type: string (nullable = true)
 |-- event_ts: timestamp (nullable = true)
 |-- ingested_at: timestamp (nullable = false)
 |-- source_topic: string (nu

In [0]:
# ──────────────────────────────────────────────────────────────
# CELL 3a │ Discover your catalog name
# ──────────────────────────────────────────────────────────────
spark.sql("SHOW CATALOGS").show()

+---------+
|  catalog|
+---------+
|  samples|
|   system|
|workspace|
+---------+



In [0]:
# ──────────────────────────────────────────────────────────────
# CELL 3b │ Create Unity Catalog Volume (run once)
# ──────────────────────────────────────────────────────────────
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.fraud_platform")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.fraud_platform.data")

print("✅ Volume ready at /Volumes/workspace/fraud_platform/data/")

✅ Volume ready at /Volumes/workspace/fraud_platform/data/


In [0]:
# ──────────────────────────────────────────────────────────────
# CELL 4 │ Write → Delta Lake Bronze  (AvailableNow trigger)
# ──────────────────────────────────────────────────────────────
bronze_query = (
    bronze_stream
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)     # ← processes all waiting messages, then stops
    .start(BRONZE_PATH)
)

bronze_query.awaitTermination()     # blocks until this batch is done

print("✅ Bronze batch complete")
print(f"   Progress: {bronze_query.lastProgress}")

✅ Bronze batch complete
   Progress: {
    "id": "fdf8a331-6ddd-49b9-813b-2ee03057d332",
    "runId": "23f0f305-43c7-49b4-8c46-57a8cf71b4e1",
    "name": null,
    "timestamp": "2026-08-20T19:46:13.483Z",
    "batchId": 1,
    "batchDuration": 7101,
    "durationMs": {
        "triggerExecution": 7100,
        "queryPlanning": 152,
        "collectSourceMetrics": 463,
        "getBatch": 1,
        "commitOffsets": 86,
        "addBatch": 4091,
        "latestOffset": 1812,
        "commitBatch": 176,
        "walCommit": 216
    },
    "eventTime": {},
    "stateOperators": [],
    "sources": [
        {
            "description": "KafkaV2[Subscribe[raw.transactions]]",
            "startOffset": "{\"raw.transactions\":{\"0\":1029,\"1\":891,\"2\":730}}",
            "endOffset": "{\"raw.transactions\":{\"0\":1742,\"1\":1483,\"2\":1209}}",
            "latestOffset": "{\"raw.transactions\":{\"0\":1742,\"1\":1483,\"2\":1209}}",
            "numInputRows": 1784,
            "inputRowsPer

In [0]:
# ──────────────────────────────────────────────────────────────
# CELL 5 │ Verify — run this any time while the stream is active
# ──────────────────────────────────────────────────────────────
df = spark.read.format("delta").load(BRONZE_PATH)

print(f"Total rows landed so far: {df.count()}")
print(f"Stream status           : {bronze_query.status}")
print()

df.select(
    "event_ts", "card_id", "merchant_name",
    "amount_local", "currency_code", "country_code",
    "_fraud_simulation_type"
).orderBy(F.col("ingested_at").desc()).show(10, truncate=False)

Total rows landed so far: 1786
Stream status           : {'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}

+-----------------------+---------+--------------------+------------+-------------+------------+----------------------+
|event_ts               |card_id  |merchant_name       |amount_local|currency_code|country_code|_fraud_simulation_type|
+-----------------------+---------+--------------------+------------+-------------+------------+----------------------+
|2026-08-20 19:31:26.636|CARD_0030|Marks & Spencer     |120.94      |GBP          |GB          |NULL                  |
|2026-08-20 19:31:23.612|CARD_0035|Supermacs           |82.27       |EUR          |IE          |NULL                  |
|2026-08-20 19:31:20.602|CARD_0021|Supermacs           |4934.68     |EUR          |IE          |high_amount           |
|2026-08-20 19:31:21.606|CARD_0014|Amazon Online       |340.31      |EUR          |IE          |NULL                  |
|2026-08-20 19:31:24.617|CA

In [0]:
# ──────────────────────────────────────────────────────────────
# CELL 6 │ Verify what landed in Bronze
# ──────────────────────────────────────────────────────────────
from pyspark.sql import functions as F

df = spark.read.format("delta").load(BRONZE_PATH)
print(f"Total rows in Bronze: {df.count()}")

df.select(
    "event_ts", "card_id", "merchant_name",
    "amount_local", "currency_code", "country_code",
    "_fraud_simulation_type"
).orderBy(F.col("ingested_at").desc()).show(10, truncate=False)

Total rows in Bronze: 1786
+-----------------------+---------+--------------------+------------+-------------+------------+----------------------+
|event_ts               |card_id  |merchant_name       |amount_local|currency_code|country_code|_fraud_simulation_type|
+-----------------------+---------+--------------------+------------+-------------+------------+----------------------+
|2026-08-20 19:31:26.636|CARD_0030|Marks & Spencer     |120.94      |GBP          |GB          |NULL                  |
|2026-08-20 19:31:23.612|CARD_0035|Supermacs           |82.27       |EUR          |IE          |NULL                  |
|2026-08-20 19:31:20.602|CARD_0021|Supermacs           |4934.68     |EUR          |IE          |high_amount           |
|2026-08-20 19:31:21.606|CARD_0014|Amazon Online       |340.31      |EUR          |IE          |NULL                  |
|2026-08-20 19:31:24.617|CARD_0017|Ryanair Online      |260.02      |EUR          |IE          |rapid_succession      |
|2026-08-20 1

In [0]:
# ──────────────────────────────────────────────────────────────
# CELL 7 │ Summary stats — run after a few batches
# ──────────────────────────────────────────────────────────────
from pyspark.sql import functions as F

df = spark.read.format("delta").load(BRONZE_PATH)

print(f"Total rows: {df.count()}")
print()

print("── By country ──")
df.groupBy("country_code").count().orderBy("count", ascending=False).show()

print("── Fraud vs Normal ──")
df.groupBy(
    F.when(F.col("_fraud_simulation_type").isNull(), "normal")
     .otherwise("suspicious")
     .alias("type")
).count().show()

print("── Fraud types breakdown ──")
df.filter(F.col("_fraud_simulation_type").isNotNull()) \
  .groupBy("_fraud_simulation_type").count().show()

print("── Amount range ──")
df.select(
    F.min("amount_local").alias("min_€"),
    F.max("amount_local").alias("max_€"),
    F.avg("amount_local").alias("avg_€")
).show()

Total rows: 1786

── By country ──
+------------+-----+
|country_code|count|
+------------+-----+
|          IE| 1424|
|          GB|  174|
|          FR|  157|
|          BR|   10|
|          RO|    9|
|          UA|    8|
|          NG|    4|
+------------+-----+

── Fraud vs Normal ──
+----------+-----+
|      type|count|
+----------+-----+
|suspicious|  104|
|    normal| 1682|
+----------+-----+

── Fraud types breakdown ──
+----------------------+-----+
|_fraud_simulation_type|count|
+----------------------+-----+
|       foreign_country|   31|
|      rapid_succession|   24|
|        suspicious_mcc|   27|
|           high_amount|   22|
+----------------------+-----+

── Amount range ──
+-----+-------+------------------+
|min_€|  max_€|             avg_€|
+-----+-------+------------------+
| 1.83|9270.56|253.35284434490487|
+-----+-------+------------------+

